# chemagent — Debugging Notebook

End-to-end pipeline using the consolidated `chemagent_mcp.py` server (16 tools).

**Preferred workflow (data stays on disk)**
```
find_datasets → load_dataset → compute_features → split_dataset
  → train_model (non-blocking) → check_training (poll)
  → plot_classification_results
```
**Shortcut (load/featurize/split synchronously, then trains in background)**
```
job = run_pipeline("data/datasets/...", algorithm="RFC", task="classification")
result = check_training(job["job_id"])   # poll every 15-30 s
```


## 1. Environment setup

In [1]:
import sys
from pathlib import Path

# Resolve workspace root (works whether cwd is notebooks/ or the repo root)
_ws_root = Path.cwd()
if not (_ws_root / "src").exists():
    _ws_root = _ws_root.parent

_servers_dir = _ws_root / "src" / "chemagent" / "servers"

for _p in [str(_ws_root), str(_servers_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Workspace root : {_ws_root}")
print(f"Servers dir    : {_servers_dir}")

Workspace root : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability
Servers dir    : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\src\chemagent\servers


## 2. Imports

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

from src.chemagent.servers.chemagent_mcp import (
    # ── Dataset tools (MCP) ────────────────────────────────────
    find_datasets,
    list_loaded_datasets,
    list_featurizers,
    load_dataset,
    dataset_status,
    compute_features,
    split_dataset,
    # ── ML tools (MCP) ─────────────────────────────────────────
    get_ml_info,
    train_model,
    check_training,
    export_predictions,
    # ── Plot tools (MCP) ───────────────────────────────────────
    plot_dataset_info,
    plot_classification_results,
    plot_regression_results,
    # ── Utility (MCP) ──────────────────────────────────────────
    run_pipeline,
    # ── Internal helpers (not MCP tools, useful for notebook) ──
    build_model_from_split_file,
    evaluate_classification,
    evaluate_regression,
)
from src.chemagent.plots import (
    set_theme,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_pr_curve,
    plot_metric_bar,
    plot_feature_importance,
    plot_class_distribution,
    plot_split_statistics,
)
import joblib, numpy as np, time


## 3. Discover available options

In [3]:
find_datasets()


{'datasets': ['chembl_activity_data_O00329_P42336.csv',
  'chembl_activity_data_O00329_P48736.csv',
  'chembl_activity_data_P42336_P48736.csv'],
 'count': 3,
 'directory': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\datasets'}

In [4]:
list_featurizers()

{'ECFP': {'parameters': {'n_bits': '2048', 'radius': '2', 'sparse': 'False'},
  'description': 'Generate ECFP (Morgan) bit-vector fingerprints from SMILES strings.'},
 'MACCS': {'parameters': {},
  'description': 'Generate 166-bit MACCS structural-key fingerprints from SMILES strings.'},
 'RDKitFP': {'parameters': {'n_bits': '2048',
   'min_path': '1',
   'max_path': '7'},
  'description': 'Generate RDKit topological (path-based) fingerprints.'},
 'AtomPairFP': {'parameters': {'n_bits': '2048'},
  'description': 'Generate atom-pair fingerprints.'},
 'TopologicalTorsionFP': {'parameters': {'n_bits': '2048'},
  'description': 'Generate topological-torsion fingerprints.'}}

In [5]:
# Returns algorithms + hyperparameter grids + recommended metrics in one call
get_ml_info()


{'algorithms': {'RFR': {'name': 'Random Forest Regressor',
   'task_type': 'regression',
   'hyperparameters': {'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]},
   'supports_multiclass': False,
   'description': 'Ensemble of decision trees for regression tasks'},
  'RFC': {'name': 'Random Forest Classifier',
   'task_type': 'classification',
   'hyperparameters': {'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]},
   'supports_multiclass': True,
   'description': 'Ensemble of decision trees for classification, handles multi-class'},
  'SVC': {'name': 'Support Vector Classifier',
   'task_type': 'classification',
   'hyperparameters': {'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']},
   'supports_multiclass': True,
   'description': 'SVM classifier with RBF/linear kern

## 4. Load dataset

In [6]:
dataset_info = load_dataset("data/datasets/chembl_activity_data_O00329_P48736.csv")
dataset_info

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'n_samples': 1277,
 'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
 'label_col': 'class_label',
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'has_smiles': True,
 'has_precomputed_features': False,
 'loaded': True,
 'next_step': "Call featurize_dataset(dataset_id='chembl_activity_data_O00329_P48736', method='ECFP', radius=2, n_bits=2048) to compute fingerprints server-side, then split_prepared_dataset().",
 'smiles_sample': ['CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(Cl)c2c(=O)n1-c1ccc(O)cc1',
  'Cc1cccc(NS(=O)(=O)c2ccc(C)c(-c3cnc(N)c(-c4cnn(C)c4)c3)c2)n1',
  'Cc1ccc(S(=O)(=O)NCC(C)(C)O)cc1-c1cnc(N)c(-c2ncco2)c1']}

In [7]:
dataset_status(dataset_info["dataset_id"])

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'loaded': True,
 'raw_data': {'n_samples': 1277,
  'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
  'label_col': 'class_label',
  'smiles_col': 'smiles',
  'id_col': None},
 'prepared': False}

## 5. Featurize (server-side — no large arrays transferred)

In [8]:
featurized = compute_features(
    dataset_info["dataset_id"],
    method="ECFP",
    radius=2,
    n_bits=2048,
)
featurized

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'method': 'ECFP',
 'n_samples': 1277,
 'n_features': 2048,
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'prepared': True,
 'next_step': "Call split_dataset('chembl_activity_data_O00329_P48736', train_size=0.7, val_size=0.0, test_size=0.3, stratified=True) to create splits."}

## 6. Split

In [9]:
data_splits = split_dataset(
    dataset_info["dataset_id"],
    train_size=0.7,
    val_size=0.0,
    test_size=0.3,
    stratified=True,
    split_type="scaffold",
)
data_splits

{'train': {'n_samples': 895,
  'indices': [457,
   380,
   1182,
   152,
   840,
   400,
   783,
   435,
   638,
   754,
   2,
   959,
   348,
   1095,
   811,
   247,
   108,
   466,
   80,
   43,
   486,
   284,
   200,
   118,
   612,
   925,
   243,
   522,
   672,
   822,
   841,
   405,
   623,
   675,
   847,
   458,
   13,
   267,
   347,
   618,
   786,
   990,
   1161,
   1204,
   1219,
   1226,
   1227,
   1239,
   1259,
   1270,
   853,
   230,
   252,
   278,
   488,
   501,
   609,
   636,
   816,
   861,
   1128,
   227,
   429,
   924,
   747,
   182,
   205,
   551,
   1114,
   1164,
   1212,
   1213,
   1216,
   1229,
   1232,
   1250,
   1255,
   1256,
   1258,
   767,
   446,
   1157,
   470,
   166,
   297,
   1060,
   299,
   681,
   1151,
   314,
   385,
   436,
   974,
   403,
   929,
   745,
   189,
   1175,
   500,
   188,
   820,
   15,
   218,
   255,
   326,
   869,
   967,
   1017,
   171,
   438,
   835,
   906,
   1027,
   1049,
   9,
   742,
   785,
   

## 6b. Dataset & split visualisation

In [10]:
# MCP equivalent (one call):
#   plot_dataset_info(dataset_info["dataset_id"], split_file_path=data_splits["saved_to"])

_split = joblib.load(data_splits["saved_to"])
_all_labels = np.concatenate([_split["train_labels"], _split["test_labels"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_class_distribution(
    _all_labels,
    title="Class Distribution (full dataset)",
    ax=axes[0],
)
plot_split_statistics(
    data_splits["statistics"],
    title="Train / Test Split",
    ax=axes[1],
)
fig.tight_layout()
plt.show()

[03/03/26 17:55:50] INFO     Using categorical units to plot a list of strings that are all         category.py:224
                             parsable as floats or dates. If these strings should be plotted as                    
                             numbers, cast to the appropriate data type before plotting.                           

                    INFO     Using categorical units to plot a list of strings that are all         category.py:224
                             parsable as floats or dates. If these strings should be plotted as                    
                             numbers, cast to the appropriate data type before plotting.                           

C:\Users\janela\AppData\Local\Temp\ipykernel_100624\4195542558.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Train model (tune → train → evaluate)

**MCP pattern** (non-blocking, used by the LLM agent):
```python
job = train_model(split_file_path=..., algorithm="RFC", task="classification")
result = check_training(job["job_id"])   # poll until status != "running"
```

**Notebook cell below** uses the internal helper `build_model_from_split_file` (blocking, convenient for direct exploration).

In [11]:
# Non-blocking MCP approach (mirrors what the LLM agent does)
job = train_model(
    split_file_path=data_splits["saved_to"],
    algorithm="RFC",
    task="classification",
    opt_metric="balanced_accuracy",
)
print("Job submitted:", job["job_id"])

while True:
    status = check_training(job["job_id"])
    if status["status"] != "running":
        break
    print(f"Training... {status['elapsed_seconds']:.1f}s elapsed")
    time.sleep(5)

model_result = status["result"]
print("Status:", status["status"])
model_result

Job submitted: 82242717-25d5-43eb-b0a4-ed46d75f9799
Training... 0.0s elapsed
Training... 5.0s elapsed
Training... 10.0s elapsed
Training... 15.1s elapsed
Training... 20.2s elapsed
Status: completed


{'algorithm': 'RFC',
 'task': 'classification',
 'cv_fold': 5,
 'opt_metric': 'balanced_accuracy',
 'best_params': {'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 200},
 'cv_best_score': 0.8480128182911649,
 'model_path': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\logs\\session_tiago_20260303_160755_bb4cb0\\models\\chembl_activity_data_O00329_P48736_scaffold_RFC.pkl',
 'hyperparameters_searched': {'n_estimators': [50, 100, 200],
  'max_features': ['sqrt', 'log2'],
  'min_samples_split': [2, 5, 10],
  'min_samples_leaf': [1, 2, 4]},
 'train_evaluation': {'target': 'train',
  'algorithm': 'RFC',
  'overall_metrics': {'MCC': 1.0,
   'Accuracy': 1.0,
   'BA': 1.0,
   'F1 macro': 1.0,
   'F1 weighted': 1.0},
  'per_class_metrics': {'Class_0': {'Precision': 1.0,
    'Recall': 1.0,
    'F1': 1.0,
    'Support': 658},
   'Class_1': {'Precision': 1.0, 'Recall': 1.0, 'F1': 

## 8. Inspect results

In [12]:
print("Best params  :", model_result["best_params"])
print("CV best score:", model_result["cv_best_score"])
print("Model saved  :", model_result["model_path"])

Best params  : {'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
CV best score: 0.8480128182911649
Model saved  : C:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\data\logs\session_tiago_20260303_160755_bb4cb0\models\chembl_activity_data_O00329_P48736_scaffold_RFC.pkl


In [13]:
model_result["train_evaluation"]

{'target': 'train',
 'algorithm': 'RFC',
 'overall_metrics': {'MCC': 1.0,
  'Accuracy': 1.0,
  'BA': 1.0,
  'F1 macro': 1.0,
  'F1 weighted': 1.0},
 'per_class_metrics': {'Class_0': {'Precision': 1.0,
   'Recall': 1.0,
   'F1': 1.0,
   'Support': 658},
  'Class_1': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 203},
  'Class_2': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 34}},
 'confusion_matrix': [[658, 0, 0], [0, 203, 0], [0, 0, 34]],
 'class_labels': [0, 1, 2]}

In [14]:
model_result["test_evaluation"]

{'target': 'test',
 'algorithm': 'RFC',
 'overall_metrics': {'MCC': 0.6293599642063776,
  'Accuracy': 0.8900523560209425,
  'BA': 0.6814848436304088,
  'F1 macro': 0.7478699461458082,
  'F1 weighted': 0.8815621776670703},
 'per_class_metrics': {'Class_0': {'Precision': 0.9003021148036254,
   'Recall': 0.9706840390879479,
   'F1': 0.9341692789968652,
   'Support': 307},
  'Class_1': {'Precision': 0.813953488372093,
   'Recall': 0.5737704918032787,
   'F1': 0.6730769230769231,
   'Support': 61},
  'Class_2': {'Precision': 0.875,
   'Recall': 0.5,
   'F1': 0.6363636363636364,
   'Support': 14}},
 'confusion_matrix': [[298, 8, 1], [26, 35, 0], [7, 0, 7]],
 'class_labels': [0, 1, 2]}

## 9. Standalone evaluation (optional)

Uses internal helpers `evaluate_classification` / `evaluate_regression` (not MCP tools — available for direct notebook use).

In [15]:
split    = joblib.load(data_splits["saved_to"])
model    = joblib.load(model_result["model_path"])
X_test   = np.array(split["test_features"])
y_test   = split["test_labels"].tolist()

y_pred   = model.predict(X_test).tolist()
y_proba  = model.predict_proba(X_test).tolist()

# internal helper — still available for custom evaluation in the notebook
evaluate_classification(
    labels=y_test,
    predictions=y_pred,
    probabilities=y_proba,
)

{'target': None,
 'algorithm': 'Model',
 'overall_metrics': {'MCC': 0.6293599642063776,
  'Accuracy': 0.8900523560209425,
  'BA': 0.6814848436304088,
  'F1 macro': 0.7478699461458082,
  'F1 weighted': 0.8815621776670703},
 'per_class_metrics': {'Class_0': {'Precision': 0.9003021148036254,
   'Recall': 0.9706840390879479,
   'F1': 0.9341692789968652,
   'Support': 307},
  'Class_1': {'Precision': 0.813953488372093,
   'Recall': 0.5737704918032787,
   'F1': 0.6730769230769231,
   'Support': 61},
  'Class_2': {'Precision': 0.875,
   'Recall': 0.5,
   'F1': 0.6363636363636364,
   'Support': 14}},
 'confusion_matrix': [[298, 8, 1], [26, 35, 0], [7, 0, 7]],
 'class_labels': [0, 1, 2]}

## 9b. Classification plots

**MCP equivalent** (loads model + split from disk, generates all figures in one call):
```python
plot_classification_results(model_result["model_path"], data_splits["saved_to"])
```

Cells below call the underlying plot functions directly for finer notebook control.

In [16]:
_n_classes  = len(set(y_test))
_is_binary  = _n_classes == 2

# binary eval → flat dict; multiclass eval → nested, scalars in "overall_metrics"
_test_metrics = model_result["test_evaluation"]
_scalar_metrics = _test_metrics.get("overall_metrics", _test_metrics)

if _is_binary:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    plot_confusion_matrix(
        y_test, y_pred,
        title="Confusion Matrix (test)", ax=axes[0],
    )
    plot_roc_curve(
        y_test, [p[1] for p in y_proba],
        title="ROC Curve (test)", ax=axes[1],
    )
    plot_pr_curve(
        y_test, [p[1] for p in y_proba],
        title="Precision-Recall Curve (test)", ax=axes[2],
    )
else:
    # Multiclass: confusion matrix + overall metric bar
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_confusion_matrix(
        y_test, y_pred,
        title=f"Confusion Matrix — {_n_classes} classes (test)", ax=axes[0],
    )
    plot_metric_bar(_scalar_metrics, title="Test Set Metrics (overall)", ax=axes[1])

fig.tight_layout()
plt.show()

C:\Users\janela\AppData\Local\Temp\ipykernel_100624\2585638552.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Evaluation metrics bar chart
# binary eval → flat dict; multiclass eval → nested, scalars in "overall_metrics"
_eval = model_result["test_evaluation"]
_scalars = _eval.get("overall_metrics", _eval)
plot_metric_bar(_scalars, title="Test Set Metrics")
plt.show()

C:\Users\janela\AppData\Local\Temp\ipykernel_100624\162163069.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Top-20 feature importances (RFC / tree-based models only)
_model = joblib.load(model_result["model_path"])
plot_feature_importance(_model, top_n=20, title="Top 20 Feature Importances")
plt.show()

C:\Users\janela\AppData\Local\Temp\ipykernel_100624\1960395262.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9c. Export predictions to CSV (MCP tool)

`export_predictions` writes a per-sample CSV with (where available):

| Column | Description |
|---|---|
| `cid` | Compound ID (from the original dataset, if present) |
| `smiles` | SMILES string (if present) |
| `true_label` | Ground-truth label |
| `predicted_label` | Model prediction |
| `prob_class_0`, `prob_class_1` | Class probabilities (classification only) |

For regression tasks `predicted_label` is replaced by `predicted_value`. A `.pkl` metrics file is saved alongside the CSV. Both land in `<session>/results/`.

In [19]:
export_result = export_predictions(
    model_path=model_result["model_path"],
    split_file_path=data_splits["saved_to"],
    task="classification",
    split="test",          # default
)
export_result

{'csv_path': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\logs\\session_tiago_20260303_160755_bb4cb0\\results\\chembl_activity_data_O00329_P48736_scaffold_RFC_test_predictions.csv',
 'metrics_path': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\logs\\session_tiago_20260303_160755_bb4cb0\\results\\chembl_activity_data_O00329_P48736_scaffold_RFC_test_metrics.pkl',
 'metrics': {'target': None,
  'algorithm': 'chembl_activity_data_O00329_P48736_scaffold_RFC',
  'overall_metrics': {'MCC': 0.6293599642063776,
   'Accuracy': 0.8900523560209425,
   'BA': 0.6814848436304088,
   'F1 macro': 0.7478699461458082,
   'F1 weighted': 0.8815621776670703},
  'per_class_metrics': {'Class_0': {'Precision': 0.9003021148036254,
    'Recall': 0.9706840390879479,
    'F1': 0.9341692789968652,
    'Support': 307},
   'Class_1': {'Precision': 0.813953488372093,
 

In [20]:
import pandas as pd
pd.read_csv(export_result["csv_path"]).head(10)


,smiles,true_label,predicted_label,prob_class_0,prob_class_1,prob_class_2
0,Cc1csc2cc(C(C)Nc3ncnc(N)c3C#N)c(Cc3cccc(CN4CCC...,1,1,0.435,0.565,0.000
1,Cc1csc2nc(C(C)Nc3ncnc4[nH]cnc34)c(-c3cccc(F)c3...,1,0,0.735,0.265,0.000
2,Cc1cc(-c2cc(-c3cc(S(=O)(=O)NCC4(CO)CCCC4)ccc3C...,0,0,1.000,0.000,0.000
3,Cc1cc(-c2cc(-c3cc(S(=O)(=O)NCC4(O)CCCC4)ccc3C)...,0,0,1.000,0.000,0.000
4,O=C(C1CCCCN1)N1CCN(c2nc(N3CCOCC3)nc(-n3c(C(F)F...,1,0,0.700,0.300,0.000
5,COc1ncc(-c2ccc3ncnc(NC4CCCN(C(=O)CN(C)C)C4)c3c...,1,0,0.880,0.115,0.005
6,CC(C)(C)NS(=O)(=O)c1cncc(-c2ccc3nc(NC(=O)NCC(=...,0,0,0.955,0.045,0.000
7,CC(C)n1nc(-c2ccc3oc(N)nc3c2)c2c(N)ncnc21,0,0,0.970,0.030,0.000
8,CC(C)n1nc(-c2ccc3oc(N)nc3c2)c2cncnc21,0,0,0.920,0.080,0.000
9,CCOC(Cn1nc(-c2ccc3oc(N)nc3c2)c2c(N)ncnc21)OCC,0,0,0.945,0.045,0.010


## 10. Scratch / exploratory

In [21]:
# Scratch cell — free-form exploration
